# The ANN index as a set of segments: incremental append, never a rebuild

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/23-segmented-ann/segmented-ann.ipynb)

Built from [`cookbook/book/chapters/23-segmented-ann/segmented-ann.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/23-segmented-ann/segmented-ann.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures. On that GPU the
# chapter runs at `full` scale, over the published data; set SCALE = "small" to
# run the seconds-long version over the committed fixtures instead.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook==0.49.1"], check=True)
SCALE = "full" if gpu else "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** `generate_embeddings` · `list_index_segments` · `search(exact=…)` ·
`refresh_embeddings` · **Theory:** append-only immutable segments merged at read time —
the shape of an LSM-tree's SSTables, never rewritten in place, only added to and later
compacted (Kleppmann 2017) — each a per-segment HNSW proximity graph
(Malkov & Yashunin 2020) · **Rail:** measurement (the segment set read back from the engine;
the approximate search checked against the engine's own exact search; the first
segment's bytes compared across an append).

A table's ANN index is a **set** of immutable segments rather than one sidecar. Adding
rows writes a new segment beside the existing ones and leaves them byte-for-byte
untouched — the index grows without rebuilding any graph — and a search fans across
every segment, merging the results under one total order. This chapter measures both
halves: a table with one segment searches exactly as a single index does, and appending
a segment leaves its neighbour untouched while making the new rows reachable.

In [ ]:
import hashlib
import tempfile
from pathlib import Path
from urllib.parse import urlparse

import jammi
import pyarrow as pa
import pyarrow.parquet as pq
from jammi_cookbook import contracts, fixtures

MODEL = fixtures.model("tiny_bert")
work = Path(tempfile.mkdtemp())
db = jammi.connect(f"file://{tempfile.mkdtemp()}")

patents = pq.read_table(fixtures.path("tiny_corpus.parquet")).select(["id", "title", "content"])
pq.write_table(patents, work / "patents.parquet")
db.add_source("patents", url=str(work / "patents.parquet"), format="parquet")
table = db.generate_embeddings(source="patents", model=MODEL, columns=["content"], key="id")
print(f"{patents.num_rows} patents embedded into {table[:48]}…")

## The segment set — asked of the engine itself

`list_index_segments(table)` returns the index's segments, one dict per segment in
`segment_id` order: its id, its bundle's path, its row count, and the version it was
written for. A single `generate_embeddings` pass writes exactly one.

In [ ]:
segments = db.list_index_segments(table)
for s in segments:
    print(f"segment {s['segment_id']}: {s['row_count']} rows  "
          f"{Path(urlparse(s['index_path']).path).name}")

In [ ]:
assert [s["segment_id"] for s in segments] == [0]
assert segments[0]["row_count"] == patents.num_rows
assert Path(urlparse(segments[0]["index_path"]).path).name.startswith(f"{table}__seg0")

The answer comes from the engine, never from opening its catalog file: an embedded
engine holds its catalog open, and a second SQLite library in the same process — which
is what Python's `sqlite3` module is — cannot see the engine's locks, so its reads can be
stale and its close can truncate a write-ahead log the engine still tracks. An empty
answer means no segments, a flat index, an unknown table, or a table this tenant cannot
see — deliberately indistinguishable, so the verb is not an existence oracle for another
tenant's table names.

## One segment: the approximate search is the exact top-*k*

`search(exact=True)` scores every vector — the true nearest neighbours. Over one segment
of an `F32` index the approximate search draws on a single graph with no over-fetch and
no rescore, so it must return the same ranking. Every row, as a query by example:

In [ ]:
K = 5
keys = [str(k) for k in patents.column("id").to_pylist()]


def ranked(key: str, exact: bool) -> list[str]:
    hits = db.search("patents", row_key=key, k=K, embedding_table=table, exact=exact)
    return hits.column("_row_id").to_pylist()


agree = sum(ranked(key, False) == ranked(key, True) for key in keys)
print(f"approximate == exact for {agree}/{len(keys)} queries (k={K})")

In [ ]:
contracts.assert_close("segmented_ann.n1.search_matches_exact_fraction", agree / len(keys))

## Appending a segment leaves its neighbour untouched

A source change reaches the table through `refresh_embeddings`, which embeds only what
changed and publishes a new version. The new rows' vectors go into a **new** segment;
segment 0 is not rebuilt. Four patents are added to the source; the refresh appends one
segment for them, and segment 0's bundle — its path, its row count, its bytes — is
exactly what it was.

In [ ]:
def digest(index_path: str) -> str:
    """One digest over every file of the segment's bundle (`{stem}.usearch`,
    `.rowmap`, `.manifest.json`), the siblings its `index_path` names."""
    base = Path(urlparse(index_path).path)
    bundle = sorted(base.parent.glob(f"{base.stem}.*"))
    assert bundle, f"no bundle files beside {base}"
    return hashlib.sha256(b"".join(p.read_bytes() for p in bundle)).hexdigest()


seg0_before = digest(segments[0]["index_path"])
added = pa.table({
    "id": [101, 102, 103, 104],
    "title": ["Superconducting qubit readout", "Protein folding with transformers",
              "Solid electrolyte interphase control", "Graph kernels for chemistry"],
    "content": [
        "A dispersive readout scheme for superconducting qubits with reduced crosstalk.",
        "Attention-based models predict protein tertiary structure from sequence alone.",
        "We engineer the solid electrolyte interphase to extend lithium battery life.",
        "Graph kernels over molecular graphs predict reaction yields in organic chemistry.",
    ],
})
pq.write_table(pa.concat_tables([patents, added.cast(patents.schema)]), work / "patents.parquet")
report = db.refresh_embeddings(table)
after = db.list_index_segments(table)
for s in after:
    print(f"segment {s['segment_id']}: {s['row_count']} rows, version {s['version']}")
print(f"refresh: {report['outcome']}, {report['inferred_rows']} rows embedded")

In [ ]:
assert report["outcome"] == "published" and report["inferred_rows"] == added.num_rows
seg0_after = next(s for s in after if s["segment_id"] == 0)
assert (seg0_after["index_path"], seg0_after["row_count"]) == (
    segments[0]["index_path"], segments[0]["row_count"])
assert digest(seg0_after["index_path"]) == seg0_before
assert [s["row_count"] for s in after if s["segment_id"] != 0] == [added.num_rows]

In [ ]:
print(f"segment 0 bytes unchanged by the append: {digest(seg0_after['index_path']) == seg0_before}")

## A search merges every segment

A query now fans across both segments and merges their candidates. Each row — old or
new — is its own nearest neighbour, found through whichever segment holds it:

In [ ]:
all_keys = keys + [str(k) for k in added.column("id").to_pylist()]
self_found = sum(ranked(key, False)[0] == key for key in all_keys)
print(f"{self_found}/{len(all_keys)} rows find themselves first "
      f"({len(keys)} in segment 0, {added.num_rows} in the new segment)")
db.close()

In [ ]:
assert self_found == len(all_keys)

## What was measured

- A freshly built table's index is one segment, and its approximate search returns the
  exact top-*k* for every row: at `N = 1` the segmented index is the single index it
  replaced, not a new special case.
- An append writes a new segment and leaves the first byte-for-byte untouched: the index
  grows by adding, never by rebuilding.
- A search merges every segment: the new rows are reachable the moment their segment is
  published, and the old ones still are.

Compaction (chapter 25) is the other half of the lifecycle: it rewrites a refreshed
table's live rows into fresh segments at the configured size, so the set does not grow
without bound.

## References

- Kleppmann, Martin (2017) *Designing Data-Intensive Applications: The Big Ideas Behind Reliable, Scalable, and Maintainable Systems* O'Reilly Media.
- Malkov, Yu A., Yashunin, Dmitry A. (2020) *Efficient and Robust Approximate Nearest Neighbor Search Using Hierarchical Navigable Small World Graphs* IEEE Transactions on Pattern Analysis and Machine Intelligence DOI 10.1109/TPAMI.2018.2889473; arXiv:1603.09320.